# 状態表現学習

状態表現学習では、未来予測を使った意思決定を、実装可能な形に分解して学びます。


## 背景と目的

良い状態表現は、制御に必要な情報を圧縮し学習を高速化します。

表現学習を理解すると、観測次元が大きい環境でも計画を成立させやすくなります。

表現の良し悪しを下流性能で検証します。


## 最初に解きたい疑問

1. `状態表現学習` は、何を学習しているのか。
2. 圧縮した `z` が、どれくらい情報を保てばよいのか。
3. 表現の良し悪しを `下流性能` で見るとは、具体的に何を比べるのか。
4. `観測予測` ができることと、良い表現であることはどう関係するのか。
5. `score_plan` が使えるようになると、何が嬉しいのか。


## 先に押さえる言葉

- `状態表現`: 観測を圧縮して、意思決定に使いやすくした内部表現です。必要な情報だけを残すことが狙いです。
- `圧縮状態`: 元の観測より低次元にまとめた状態です。計算を軽くしながら、重要情報を保つために使います。
- `下流性能`: その表現を使った後段のタスクの成績です。復元だけでなく制御や計画で効くかを見ます。
- `情報保持`: 予測や制御に必要な情報が表現内に残っていることです。削りすぎると後段の性能が落ちます。


## 実行前の見取り図

1. `z_next` が圧縮状態として 1 つの数値で出るか。
2. `obs_next` が `position` と `velocity` に復元できるか。
3. `scores` が計画候補ごとに違う値になり、表現が計画判断に使われているか。


## 出力の読み方

- `scores` が計画判断に使われている読み方。
- `z_next` が 1 つの数値でも何を表すかの見方。


## つまずきやすい点

- 「表現が良い」の判定を、復元誤差だけでなく下流の計画成績で見る理由。
- どの情報が削れてはいけないのか、という視点の説明。
- 復元誤差だけでは不十分で下流性能を見る理由。
- 削ってはいけない情報の見方。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- ここでは学習済み encoder の代わりに手書き z を使う近似である点。


## このノートの位置づけ

ここでは学習済み encoder の代わりに、手で置いた圧縮状態 `z` を使って「良い表現が下流予測や計画にどう効くか」を先に見ています。実際の状態表現学習では、encoder/decoder と再構成損失、予測損失、あるいは対比損失を使って `z` をデータから学習します。


## 検証 1: 表現学習の遷移初期化

状態表現学習では圧縮状態の可用性が重要なので、簡易遷移から始めます。


In [ ]:
z_t = 0.12
a_t = 0.5
A, B = 0.91, 0.19
z_next = A * z_t + B * a_t
print('task = state-representation-learning')
print('z_next =', round(z_next, 6))

このzをどれだけ情報保持できるかが論点になります。



## 検証 2: 観測予測を作る

次に、潜在状態から観測を復元する写像を作ります。状態推定と観測再現の役割分担をコードで掴みます。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、遷移誤差と観測誤差を分離して調整できます。



## 定義の確認

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$


## 検証 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1ステップでは見えない誤差累積を把握するためです。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、遷移モデルの安定性や状態表現の情報量不足を疑います。



## 検証 4: 計画候補を比較する

次に、複数の行動列を比較して、どの計画が望ましいかを評価します。モデルベース強化学習の中心操作です。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

計画評価が可能になると、実環境での試行回数を抑えた探索がしやすくなります。



## 検証 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。世界モデルは『予測できる範囲』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの遷移条件でモデルが弱いかを特定しやすくなります。



## まとめ

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 1ステップ誤差だけで安心してしまう
2. 長期予測の誤差爆発を監視しない
3. 状態表現が不足しているのにモデルだけ調整する
